# 第10章: 事前学習済み言語モデル（GPT型）

本章では、GPT型（Transformerのデコーダ型）の事前学習済みモデルを利用して、言語生成、評判分析器（ポジネガ分類器）の構築、ファインチューニング、強化学習などに取り組む。

# Chương 10: Mô hình ngôn ngữ đã tiền huấn luyện (kiểu GPT)

Trong chương này, bạn sẽ sử dụng các mô hình ngôn ngữ dạng GPT (Transformer Decoder) để thực hiện:

- sinh văn bản (text generation)
- phân tích cảm xúc (sentiment analysis)
- fine-tuning
- preference tuning / RLHF

Nếu chương 9 là BERT thì chương 10 là GPT.

## 90. 次単語予測

“The movie was full of"に続くトークン（トークン列ではなく一つのトークンであることに注意せよ）として適切なもの上位10個と、その確率（尤度）を求めよ。ただし、言語モデルへのプロンプトがどのようなトークン列に変換されたか、確認せよ。

## 90. Dự đoán token tiếp theo

Cho prompt:
```
"The movie was full of"
```
Hãy tìm:

- 10 token có xác suất cao nhất xuất hiện tiếp theo
- xác suất của từng token

Ngoài ra:

- kiểm tra prompt được tokenizer chuyển thành chuỗi token như thế nào

In [1]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [2]:
text = "The movie was full of"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

probs = torch.softmax(logits[0, -1], dim=-1)

values, indices = torch.topk(probs, k=10)

for prob, idx in zip(values, indices):
    token = tokenizer.decode([idx])
    print(token, prob.item())

 action 0.08349609375
 suspense 0.07373046875
 ______ 0.06494140625
 __ 0.03955078125
 violence 0.03076171875
 excitement 0.03076171875
 humor 0.02392578125
 ____ 0.0164794921875
 tension 0.0164794921875
 surprises 0.0145263671875


## 91. 続きのテキストの予測

“The movie was full of"に続くテキストを複数予測せよ。このとき、デコーディングの方法や温度パラメータ（temperature）を変えながら、予測される複数のテキストの変化を観察せよ。

## 91. Sinh phần tiếp theo của văn bản

Cho prompt:
```
"The movie was full of"
```
Hãy sinh nhiều đoạn văn bản tiếp theo.

Thử thay đổi:

- decoding strategy
- temperature

và quan sát kết quả.

Ví dụ:

Greedy Decoding

- Luôn chọn token xác suất cao nhất.

Sampling

- Lấy mẫu theo phân phối xác suất.

Temperature
- thấp → bảo thủ
- cao → sáng tạo hơn

In [3]:
def generate_text(prompt, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(**inputs, max_new_tokens=100, **kwargs)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
prompt = "The movie was full of"

print("Greedy")
print(generate_text(prompt, do_sample=False))

print("\nTemp=0.3")
print(generate_text(prompt, do_sample=True, temperature=0.3))

print("\nTemp=1.0")
print(generate_text(prompt, do_sample=True, temperature=1.0))

print("\nTemp=2.0")
print(generate_text(prompt, do_sample=True, temperature=2.0))

print("\nTop-k Sampling")
print(generate_text(prompt, do_sample=True, top_k=50, temperature=1.0))

print("\nTop-p (Nucleus Sampling)")
print(generate_text(prompt, do_sample=True, top_p=0.9, temperature=1.0))

Greedy
The movie was full of action and excitement, but the plot was confusing.  Given that the answer to a question is "bored", what is a question that could have been asked before this statement?  A: How did the person feel about the movie? B: What was the main reason for the confusion in the plot? C: Did the person enjoy watching the movie? D: Was the movie entertaining?
Given the context provided, the most appropriate question that could have led to the answer "bored" would

Temp=0.3
The movie was full of ________ and the audience laughed heartily. A．humor B．joke C．funny D．amusement
A 解析: 本题考查名词辨析。句意为：这部电影充满了幽默，观众哄堂大笑。humor“幽默”，是不可数名词；joke“玩笑”；funny“有趣的”；amusement“娱乐”。根据句意可知答案选A。

19、在进行施工成本控制时，需要对实际

Temp=1.0
The movie was full of action scenes, which I found to be ____. 
A. Very boring
B. Very exciting
Answer:

B

When the wind direction is stable and does not change, we call it a stable wind. Determine if this statement is true or false: A. True; B. False.
Answer:

B

Does the f

## 92. 予測されたテキストの確率を計算

“The movie was full of"に続くテキストを予測し、生成された各単語の尤度を表示せよ（生成されるテキストが長いと出力が読みにくくなるので、適当な長さで生成を打ち切るとよい）。

## 92. Tính xác suất của văn bản sinh ra

Từ prompt:
```
"The movie was full of"
```
Hãy sinh tiếp một đoạn văn.

Sau đó hiển thị:

- xác suất của từng token được sinh ra

Ví dụ:
```
The      0.95
movie    0.88
was      0.99
...
```
Để dễ đọc có thể giới hạn độ dài câu sinh ra.

In [5]:
import torch

prompt = "The movie was full of"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False, temperature=1.0,
                         return_dict_in_generate=True, output_scores=True)

In [6]:
generated_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)

print(generated_text)

The movie was full of action and excitement, but the plot was confusing.  Given that the answer to a question is "


In [7]:
input_length = inputs["input_ids"].shape[1]

generated_ids = outputs.sequences[0][input_length:]

In [8]:
import torch.nn.functional as F

for token_id, logits in zip(generated_ids, outputs.scores):
    probs = F.softmax(logits, dim=-1)

    prob = probs[0, token_id].item()

    token = tokenizer.decode([token_id])

    print(
        f"Token: {repr(token):15} "
        f"Prob: {prob:.6f}"
    )

Token: ' action'       Prob: 0.083513
Token: ' and'          Prob: 0.520695
Token: ' excitement'   Prob: 0.293094
Token: ','             Prob: 0.453134
Token: ' but'          Prob: 0.202907
Token: ' the'          Prob: 0.208163
Token: ' plot'         Prob: 0.212847
Token: ' was'          Prob: 0.386125
Token: ' confusing'    Prob: 0.118595
Token: '.'             Prob: 0.492159
Token: ' '             Prob: 0.288461
Token: ' Given'        Prob: 0.715890
Token: ' that'         Prob: 0.559833
Token: ' the'          Prob: 0.916436
Token: ' answer'       Prob: 0.914332
Token: ' to'           Prob: 0.999603
Token: ' a'            Prob: 0.887812
Token: ' question'     Prob: 0.989580
Token: ' is'           Prob: 0.995421
Token: ' "'            Prob: 0.989276


In [9]:
# version 2 - def

def generate_text(prompt, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(**inputs, max_new_tokens=20, **kwargs,
                             return_dict_in_generate=True, output_scores=True)

    return inputs, outputs, tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
import torch.nn.functional as F

def compute_prob(inputs, outputs):
    input_length = inputs["input_ids"].shape[1]

    generated_ids = outputs.sequences[0][input_length:]

    scores = outputs.scores

    for i, token_id in enumerate(generated_ids):
        logits = scores[i]

        probs = F.softmax(logits, dim=-1)

        prob = probs[0, token_id].item()

        token = tokenizer.decode([token_id])

        print(f"{token:15s} {prob:.6f}")

In [11]:
prompt = "The movie was full of"

print("Greedy")
inputs, outputs, text = generate_text(prompt, do_sample=False)
print(text)
compute_prob(inputs, outputs)

Greedy
['The movie was full of action and excitement, but the plot was confusing.  Given that the answer to a question is "']
 action         0.083513
 and            0.520695
 excitement     0.293094
,               0.453134
 but            0.202907
 the            0.208163
 plot           0.212847
 was            0.386125
 confusing      0.118595
.               0.492159
                0.288461
 Given          0.715890
 that           0.559833
 the            0.916436
 answer         0.914332
 to             0.999603
 a              0.887812
 question       0.989580
 is             0.995421
 "              0.989276


In [12]:
print("\nTemp=0.3")
inputs, outputs, text = generate_text(prompt, do_sample=True, temperature=0.3)
print(text)
compute_prob(inputs, outputs)


Temp=0.3
["The movie was full of suspense and excitement, but I couldn't help feeling a little bored. Which of the following options best"]
 suspense       0.314847
 and            1.000000
 excitement     0.376047
,               1.000000
 but            0.323224
 I              0.222700
 couldn         0.147247
't              1.000000
 help           1.000000
 feeling        1.000000
 a              0.397315
 little         0.302940
 bored          0.302941
.               1.000000
 Which          1.000000
 of             0.197575
 the            1.000000
 following      1.000000
 options        0.777300
 best           1.000000


In [13]:
print("\nTemp=2.0")
inputs, outputs, text = generate_text(prompt, do_sample=True, temperature=2.0)
print(text)
compute_prob(inputs, outputs)


Temp=2.0
['The movie was full of suspense because the film______(direct) was not easy.\n\ndirected\n\n16世纪的西']
 suspense       0.110549
 because        0.053864
 the            0.240668
 film           0.043298
____            0.055267
__(             0.090016
direct          0.082646
)               0.536416
 was            0.053986
 not            0.101097
 easy           0.068028
.

             0.106531
direct          0.235173
ed              0.434152


              0.508136
1               0.072834
6               0.076371
世纪              0.090385
的               0.146511
西               0.064710


In [14]:
print("\nTop-k Sampling")
inputs, outputs, text = generate_text(prompt, do_sample=True, top_k=50, temperature=1.0)
print(text)
compute_prob(inputs, outputs)


Top-k Sampling
['The movie was full of excitement and anticipation for fans who had been waiting years to see their favorite action star in a live performance']
 excitement     0.061334
 and            0.479675
 anticipation   0.025566
 for            0.102286
 fans           0.038871
 who            0.132010
 had            0.297937
 been           0.584304
 waiting        0.622459
 years          0.289126
 to             0.542896
 see            1.000000
 their          0.195892
 favorite       0.834436
 action         0.025459
 star           0.115971
 in             0.248623
 a              0.466839
 live           0.057940
 performance    0.404052


In [15]:
print("\nTop-p (Nucleus Sampling)")
inputs, outputs, text = generate_text(prompt, do_sample=True, top_p=0.9, temperature=1.0)
print(text)
compute_prob(inputs, outputs)


Top-p (Nucleus Sampling)
['The movie was full of suspense and I felt_______when I watched it. ____\nrelaxed\nexcited\nn']
 suspense       0.161196
 and            0.486026
 I              0.137278
 felt           0.045904
____            0.030411
___             0.193343
when            0.105286
 I              0.058089
 watched        0.540851
 it             1.000000
.               0.361442
 __             0.059910
__
             1.000000
rel             0.015229
axed            0.508825

               1.000000
exc             0.606117
ited            1.000000

               1.000000
n               0.178199


## 93. パープレキシティ

適当な文を準備して、事前学習済み言語モデルでパープレキシティを測定せよ。例えば、

+ The movie was full of surprises
+ The movies were full of surprises
+ The movie were full of surprises
+ The movies was full of surprises

の4文に対して、パープレキシティを測定して観察せよ（最後の2つの文は故意に文法的な間違いを入れた）。

## 93. Perplexity

Chuẩn bị một vài câu:

- The movie was full of surprises

- The movies were full of surprises

- The movie were full of surprises

- The movies was full of surprises

Trong đó:

- 2 câu đầu đúng ngữ pháp
- 2 câu cuối sai ngữ pháp

Hãy tính:

- và quan sát sự khác biệt.

Ý tưởng:

- câu tự nhiên → perplexity thấp
- câu kỳ quặc → perplexity cao

In [16]:
import torch
import torch.nn.functional as F

def perplexity(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model(input_ids)

    logits = outputs.logits

    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    log_probs = F.log_softmax(shift_logits, dim=-1)

    token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

    neg_log_likelihood = -token_log_probs.mean()
    ppl = torch.exp(neg_log_likelihood)

    return ppl.item()

In [17]:
texts = [
    "The movie was full of surprises",
    "The movies were full of surprises",
    "The movie were full of surprises",
    "The movies was full of surprises",
]

for t in texts:
    print(t)
    print("PPL:", perplexity(model, tokenizer, t))
    print()

The movie was full of surprises
PPL: 90.0

The movies were full of surprises
PPL: 296.0

The movie were full of surprises
PPL: 568.0

The movies was full of surprises
PPL: 536.0



In [18]:
# version 2
import torch
import torch.nn.functional as F
import math

def compute_ppl(text):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    loss = outputs.loss
    ppl = math.exp(loss.item())

    return ppl

In [19]:
texts = [
    "The movie was full of surprises",
    "The movies were full of surprises",
    "The movie were full of surprises",
    "The movies was full of surprises",
]

for t in texts:
    print(t)
    print("PPL:", compute_ppl(t))
    print()

The movie was full of surprises
PPL: 90.32274926333818

The movies were full of surprises
PPL: 290.9297633113324

The movie were full of surprises
PPL: 574.4864575080114

The movies was full of surprises
PPL: 527.5344787476887



## 94. チャットテンプレート

"What do you call a sweet eaten after dinner?"という問いかけに対する応答を生成するため、チャットテンプレートを適用し、言語モデルに与えるべきプロンプトを作成せよ。また、そのプロンプトに対する応答を生成し、表示せよ。

## 94. Chat Template

Cho câu hỏi:
```
"What do you call a sweet eaten after dinner?"
```
Hãy:

- áp dụng chat template
- tạo prompt thực sự gửi vào mô hình

Ví dụ:
```
<user>
What do you call a sweet eaten after dinner?
```
Sau đó:

- sinh câu trả lời
- hiển thị kết quả

In [20]:
import torch

def ask_model(question, max_new_tokens=50):
    messages = [{"role": "user", "content": question}]

    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print(ask_model("What do you call a sweet eaten after dinner?"))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What do you call a sweet eaten after dinner?
assistant
A sweet eaten after dinner is called an "after-dinner treat" or simply an "after-dinner snack." It's often enjoyed as a way to satisfy the appetite and provide a pleasant end to a meal.


## 95. マルチターンのチャット

問題94で生成された応答に対して、追加で"Please give me the plural form of the word with its spelling in reverse order."と問いかけたときの応答を生成・表示せよ。また、その時に言語モデルに与えるプロンプトを確認せよ。

## 95. Hội thoại nhiều lượt

Tiếp tục cuộc hội thoại ở bài 94.

Người dùng hỏi thêm:
```
Please give me the plural form of the word with its spelling in reverse order.
```
Hãy:

- sinh câu trả lời
- hiển thị prompt đầy đủ gửi vào mô hình

Mục tiêu:

- hiểu cách chatbot duy trì lịch sử hội thoại.

In [21]:
import torch

# round 1

messages = [
    {
        "role": "user",
        "content": "What do you call a sweet eaten after dinner?"
    }
]

inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id)

response1 = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(response1)

A sweet eaten after dinner is called an "after-dinner treat" or simply an "after-dinner snack." It's often enjoyed as a way to satisfy the appetite and provide a pleasant end to a meal.


In [22]:
# round 2

messages.append(
    {
        "role": "assistant",
        "content": response1
    }
)

messages.append(
    {
        "role": "user",
        "content": "Please give me the plural form of the word with its spelling in reverse order."
    }
)

In [23]:
# prompt

prompt_text = tokenizer.apply_chat_template(messages, tokenize=False,add_generation_prompt=True
)

print("Prompt:")
print(prompt_text)

Prompt:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What do you call a sweet eaten after dinner?<|im_end|>
<|im_start|>assistant
A sweet eaten after dinner is called an "after-dinner treat" or simply an "after-dinner snack." It's often enjoyed as a way to satisfy the appetite and provide a pleasant end to a meal.<|im_end|>
<|im_start|>user
Please give me the plural form of the word with its spelling in reverse order.<|im_end|>
<|im_start|>assistant



In [24]:
# response 2

inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id)

response2 = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],
                             skip_special_tokens=True)

print(response2)

The plural form of the word with its spelling in reverse order would be "sweets," since "sweet" spelled backwards is "etsw."


## 96. プロンプトによる感情分析

事前学習済み言語モデルで感情分析を行いたい。テキストを含むプロンプトを事前学習済み言語モデルに与え、（ファインチューニングは行わずに）テキストのポジネガを予測するという戦略で、[SST-2](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)の開発データにおける正解率を測定せよ。

## 96. Sentiment Analysis bằng Prompt

Không fine-tune.

Chỉ dùng prompting.

Ví dụ:
```
Review:
The movie was wonderful.

Sentiment:
```
Cho mô hình tự điền:
```
Positive
```
hoặc
```
Negative
```

Sau đó:

- chạy trên SST-2 dev set
- tính accuracy

Đây là:
```
In-context learning / Prompt-based classification
```

In [25]:
!wget https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
!unzip SST-2.zip

--2026-06-20 07:58:52--  https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.81, 13.249.182.39, 13.249.182.62, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.81|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7439277 (7.1M) [application/zip]
Saving to: ‘SST-2.zip’

SST-2.zip           100%[===================>]   7.09M  --.-KB/s    in 0.1s    

2026-06-20 07:58:53 (47.3 MB/s) - ‘SST-2.zip’ saved [7439277/7439277]

Archive:  SST-2.zip
   creating: SST-2/
  inflating: SST-2/dev.tsv           
   creating: SST-2/original/
  inflating: SST-2/original/README.txt  
  inflating: SST-2/original/SOStr.txt  
  inflating: SST-2/original/STree.txt  
  inflating: SST-2/original/datasetSentences.txt  
  inflating: SST-2/original/datasetSplit.txt  
  inflating: SST-2/original/dictionary.txt  
  inflating: SST-2/original/original_rt_snippets.txt  
  inflating: SST-2/original/s

In [26]:
import pandas as pd

dev_df = pd.read_csv("SST-2/dev.tsv",sep="\t")

print(dev_df.head())
print(len(dev_df))

                                            sentence  label
0    it 's a charming and often affecting journey .       1
1                 unflinchingly bleak and desperate       0
2  allows us to hope that nolan is poised to emba...      1
3  the acting , costumes , music , cinematography...      1
4                  it 's slow -- very , very slow .       0
872


In [27]:
import torch

def predict_sentiment(text):

    prompt = f"""
    You are a sentiment classifier.

    Review:
    {text}

    Output exactly one word:

    positive
    negative
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    answer = answer.strip().lower()

    if "positive" in answer:
        return 1

    return 0

In [28]:
# test
print(predict_sentiment("the movie was fantastic"))

print(predict_sentiment("the movie was terrible"))

1
0


In [29]:
def debug_prediction(text):

    prompt = f"""
    You are a sentiment classifier.

    Review:
    {text}

    Output exactly one word:

    positive
    negative
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print(answer)

In [30]:
debug_prediction("the movie was fantastic")

positive


In [32]:
correct = 0

for _, row in dev_df.head(20).iterrows():
    pred = predict_sentiment(row["sentence"])

    if pred == row["label"]:
        correct += 1

print("Accuracy:", correct / 20)

Accuracy: 0.95


In [ ]:
correct = 0

for _, row in dev_df.head(20).iterrows():
    pred = predict_sentiment(row["sentence"])

    if pred == row["label"]:
        correct += 1

print("Accuracy:", correct / 20)

In [35]:
#

import torch

device = next(model.parameters()).device

print(device)

cuda:0


In [37]:
correct = 0

for _, row in dev_df.head(20).iterrows():
    pred = predict_sentiment(row["sentence"])

    if pred == row["label"]:
        correct += 1

print("Accuracy:", correct / 20)

Accuracy: 0.95


In [38]:
from tqdm.auto import tqdm

correct = 0
total = len(dev_df)

for _, row in tqdm(dev_df.iterrows(), total=total):
    pred = predict_sentiment(row["sentence"])

    if pred == row["label"]:
        correct += 1

accuracy = correct / total

print()
print(f"Correct : {correct}")
print(f"Total   : {total}")
print(f"Accuracy: {accuracy:.4f}")

  0%|          | 0/872 [00:00<?, ?it/s]


Correct : 812
Total   : 872
Accuracy: 0.9312


## 97. 埋め込みに基づく感情分析

事前学習済み言語モデルでテキストをベクトルで表現（エンコード）し、そのベクトルにフィードフォワード層を通すことで極性ラベルを予測するモデルを学習せよ。

## 97. Sentiment Analysis bằng Embedding

Sử dụng GPT như một encoder.

Quy trình:
```
Text
 ↓
GPT embedding
 ↓
Feed Forward Network
 ↓
Positive / Negative
```
Huấn luyện classifier trên embedding đó.

In [45]:
import pandas as pd

train_df = pd.read_csv("SST-2/train.tsv",sep="\t")

dev_df = pd.read_csv("SST-2/dev.tsv", sep="\t")

print(len(train_df))
print(len(dev_df))

67349
872


In [46]:
from torch.utils.data import Dataset

class SSTDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        text = self.df.iloc[idx]["sentence"]
        label = self.df.iloc[idx]["label"]

        return text, float(label)

In [62]:
def collate_fn(batch):

    texts = [x[0] for x in batch]
    labels = [x[1] for x in batch]

    encodings = tokenizer(texts, padding="max_length", max_length=64, truncation=True, return_tensors="pt")

    labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return encodings, labels

In [64]:
from torch.utils.data import DataLoader

train_loader = DataLoader(SSTDataset(train_df), batch_size=32, shuffle=True, collate_fn=collate_fn)

dev_loader = DataLoader(SSTDataset(dev_df), batch_size=32, shuffle=False, collate_fn=collate_fn)

In [65]:
import torch
import torch.nn as nn

class GPTClassifier(nn.Module):

    def __init__(self, gpt_model):
        super().__init__()

        self.gpt = gpt_model

        for p in self.gpt.parameters():
            p.requires_grad = False

        hidden_size = self.gpt.config.hidden_size

        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.gpt(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        hidden = outputs.hidden_states[-1]

        sentence_vector = hidden.mean(dim=1)

        logits = self.linear(outputs.last_hidden_state.mean(dim=1).float())

        return logits

In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classifier = GPTClassifier(model.model)

classifier = classifier.to(device)

In [67]:
for param in model.parameters():
    param.requires_grad = False

In [68]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(classifier.linear.parameters(), lr=1e-3)

In [ ]:
from tqdm.auto import tqdm

epochs = 1

for epoch in range(epochs):

    classifier.train()

    total_loss = 0

    for encodings, labels in tqdm(train_loader):

        encodings = {k: v.to(device) for k, v in encodings.items()}

        labels = labels.to(device)

        optimizer.zero_grad()

        logits = classifier(encodings["input_ids"], encodings["attention_mask"])

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"{total_loss:.4f}"
    )

In [ ]:
classifier.eval()

correct = 0
total = 0

with torch.no_grad():
    for encodings, labels in dev_loader:
        encodings = {k: v.to(device) for k, v in encodings.items()}

        labels = labels.to(device)

        logits = classifier(encodings["input_ids"], encodings["attention_mask"])

        preds = (torch.sigmoid(logits) > 0.5).float()

        correct += (preds == labels).sum().item()

        total += labels.size(0)

print(f"Accuracy: {correct / total:.4f}")

## 98. ファインチューニング

問題96のプロンプトに対して、正解の感情ラベルをテキストの応答として返すように事前学習済みモデルをファインチューニングせよ。

## 98. Fine-tuning

Bài 96 chỉ prompting.

Bây giờ hãy fine-tune GPT.

Ví dụ:

Input:
```
Review:
The movie was wonderful.

Sentiment:
```
Output:
```
Positive
```
Mô hình được huấn luyện để sinh đúng label.

## 99. 選好チューニング

問題96のプロンプトに対して、正解の感情ラベルを含むテキストを望ましい応答、間違った感情ラベルを含むテキストを望ましくない応答として、事前学習済み言語モデルを選好チューニング (preference tuning) を実施せよ。選好チューニングのアルゴリズムとしては、近傍方策最適化 (PPO: Proximal Policy Optimization) や直接選好最適化 (DPO: Direct Preference Optimization) などが考えられる。

## 99. Preference Tuning

Tiếp tục bài 96.

Ví dụ:

Prompt:
```
Review:
The movie was wonderful.

Sentiment:
```
Response tốt:
```
Positive
```
Response xấu:
```
Negative
```
Huấn luyện mô hình để:

- thích response tốt
- không thích response xấu

Có thể dùng:

- PPO - Proximal Policy Optimization

- DPO - Direct Preference Optimization

Đây là kỹ thuật cốt lõi của RLHF.